In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from allensdk.brain_observatory.behavior.behavior_project_cache.\
    behavior_neuropixels_project_cache \
    import VisualBehaviorNeuropixelsProjectCache

%matplotlib inline

In [2]:
# Update this to a valid directory in your filesystem. This is where the data will be stored.
cache_dir = './data/'
cache = VisualBehaviorNeuropixelsProjectCache.from_local_cache(cache_dir=cache_dir) #from_s3_cache(cache_dir=cache_dir)

# get the metadata tables
units_table = cache.get_unit_table()
channels_table = cache.get_channel_table()
probes_table = cache.get_probe_table()
behavior_sessions_table = cache.get_behavior_session_table()
ecephys_sessions_table = cache.get_ecephys_session_table()

c:\Users\matth\anaconda3\envs\AMP3-Spring-2026-Project\Lib\site-packages\allensdk\api\cloud_cache\cloud_cache.py:547: OutdatedManifestWarning: You are loading visual-behavior-neuropixels_project_manifest_v0.5.0.json. A more up to date version of the dataset -- visual-behavior-ophys_project_manifest_v1.1.0.json -- exists online. To see the changes between the two versions of the dataset, run
VisualBehaviorNeuropixelsProjectCache.compare_manifests('visual-behavior-neuropixels_project_manifest_v0.5.0.json', 'visual-behavior-ophys_project_manifest_v1.1.0.json')
To load another version of the dataset, run
VisualBehaviorNeuropixelsProjectCache.load_manifest('visual-behavior-ophys_project_manifest_v1.1.0.json')
  warnings.warn(msg, OutdatedManifestWarning)


In [3]:
session_id = 1065437523 #1064644573
session = cache.get_ecephys_session(
            ecephys_session_id=session_id)

c:\Users\matth\anaconda3\envs\AMP3-Spring-2026-Project\Lib\site-packages\hdmf\spec\namespace.py:772: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.6.0-alpha, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


Load Relevant Data

In [4]:
# Get Neural Data
units = session.get_units()
channels = session.get_channels()
unit_channels = units.merge(channels, left_on='peak_channel_id', right_index=True)

#first let's sort our units by depth
unit_channels = unit_channels.sort_values('probe_vertical_position', ascending=False)

#now we'll filter them
good_unit_filter = ((unit_channels['snr']>1)&
                    (unit_channels['isi_violations']<1)&
                    (unit_channels['firing_rate']>0.1))
good_units = unit_channels.loc[good_unit_filter]


# Unit info
unit_indices = np.array(good_units.index)
spike_times = dict([(i,session.spike_times[i]) for i in unit_indices])
structures = dict(good_units.structure_acronym)

# Stimulus Data
stimulus_presentations = session.stimulus_presentations
# Active stimulus
active_stimulus_presentations = stimulus_presentations[stimulus_presentations["active"]]
# Active Stimulus: Time, Name, Is Change
onset_times = active_stimulus_presentations['start_time'].values
image_names = active_stimulus_presentations['image_name'].values
image_is_changes = active_stimulus_presentations['is_change'].values


# Behavioral Data
eye_tracking = session.eye_tracking
running_speed = session.running_speed
licks = session.licks

Pre Compute Bins

In [ ]:
# Set bin size (~10 ms)
bin_size = 0.01
# Set # of bins (~10=100ms=0.1 sec)
num_bins = 20

# Create bin times relative to image onset (0ms, 10ms, 20ms, ..., 90ms)
bins_times_per_onset = np.linspace(0, (num_bins-1)*bin_size, num_bins)

# Create bin start and end times for entire active session
bin_start_times = []
bin_end_times = []

# For each image onset
for onset_time in onset_times:
    # Add bin start and end times
    bin_start_times += list(onset_time + bins_times_per_onset)
    bin_end_times += list(onset_time + bins_times_per_onset + bin_size)

# Convert them to numpy arrays (for efficiency)
bin_start_times = np.array(bin_start_times)
bin_end_times = np.array(bin_end_times)

Get Binned Spike Counts

In [38]:
# Get the # of image flashes and # of units
num_image_flashes = len(onset_times)
num_units = len(spike_times)

# Create data structure for binned spike counts
hist = np.zeros((num_image_flashes, num_bins, num_units))

# For each unit
for k, unit_idx in enumerate(unit_indices):
    # Select corresponding unit's spike times
    unit_spike_times = spike_times[unit_idx]

    # Find the correct indices of the unit spike times for each bin
    start_indices = np.searchsorted(unit_spike_times, bin_start_times)
    stop_indices = np.searchsorted(unit_spike_times, bin_end_times)

    # Get the counts for each bin
    counts = stop_indices - start_indices

    # Store them in our handy-dandy "histogram"
    hist[:,:,k] = counts.reshape(num_image_flashes, num_bins)

What Brain Region Each Acronym is Part Of

In [65]:
# Map Structure Acronym to Brain Region
acronym2region = {
    "APN": "Midbrain",
    "CA1": "Hippo",
    "CA3": "Hippo",
    "DG": "Hippo",
    "Eth": "Thalamus",
    "HPF": "Hippo",
    "LP": "Thalamus",
    "MB": "Midbrain",
    "MGd": "Midbrain",
    "MGm": "Midbrain",
    "MGv": "Midbrain",
    "MRN": "Midbrain",
    "NB": "UNKNOWN", # idk
    "NOT": "Midbrain",
    "PIL": "Thalamus",
    "POL": "Thalamus",
    "POST": "Hippo",
    "ProS": "Hippo",
    "SUB": "Hippo",
    "TH": "Thalamus",
    "VISal": "VIS",
    "VISam": "VIS",
    "VISl": "VIS",
    "VISp": "VIS",
    "VISpm": "VIS",
    "VISpm": "VIS",
    "VISrl": "VIS",
    "root": "UNKNOWN", # idk

    "SGN": "Thalamus",
    "PoT": "Thalamus",
    "PP": "Thalamus",
    "RN": "Midbrain", # i think
    "LT": "Midbrain",
}